In [2]:
from pathlib import Path
import os
ROOT = Path().resolve()
print("Project root:", ROOT)
os.chdir(Path(os.getcwd()).parent)  # von /notebooks nach Projekt-Root springen
print("Working dir:", Path().absolute())

Project root: C:\Users\maumo\OneDrive\Dokumente\rag_project\notebooks
Working dir: c:\Users\maumo\OneDrive\Dokumente\rag_project


In [ ]:
from processing import read_pdf, chunk_documents  # evtl. später für Tests
from langchain_community.embeddings import HuggingFaceBgeEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_community.chat_models import ChatOllama

# Embeddings wieder initialisieren (gleich wie in 02_build_index)
embedder = HuggingFaceBgeEmbeddings(
    model_name="BAAI/bge-m3",
    encode_kwargs={"normalize_embeddings": True},
)
print("✅ Embedder initialisiert: BAAI/bge-m3")

# Pfad zum bestehenden Chroma-Index
CHROMA_DIR = ROOT / "artifacts" / "chroma_din"
print("Lade Vectorstore aus:", CHROMA_DIR)

vs = Chroma(
    embedding_function=embedder,
    persist_directory=str(CHROMA_DIR),
)

retriever = vs.as_retriever(search_kwargs={"k": 4})
print("✅ Vectorstore geladen & Retriever initialisiert.")


✅ Embedder initialisiert: BAAI/bge-m3
Lade Vectorstore aus: C:\Users\maumo\OneDrive\Dokumente\rag_project\artifacts\chroma_din
✅ Vectorstore geladen & Retriever initialisiert.


In [4]:
def answer_with_rag(
    query: str,
    vs,
    model_name: str = "llama3.1:8b",  # oder "llama4:8b" – was du in Ollama hast
    k: int = 4,
):
    """
    Einfacher RAG-Helper:
    - holt k relevante Chunks aus dem Vectorstore
    - baut einen Kontext
    - ruft ein lokales LLM (Ollama) auf
    - gibt Antwort + Quellen zurück
    """
    retriever = vs.as_retriever(search_kwargs={"k": k})
    docs = retriever.invoke(query)

    # Kontext aus den gefundenen Chunks bauen
    context_blocks = []
    for d in docs:
        src = d.metadata.get("source")
        page = d.metadata.get("page")
        context_blocks.append(
            f"[Quelle: {src}, Seite {page}]\n{d.page_content}"
        )

    context = "\n\n".join(context_blocks)

    prompt = f"""
Du bist ein Assistent für Fragen zu technischen Normen.
Nutze ausschließlich den bereitgestellten Kontext, um die Frage zu beantworten.
Wenn etwas nicht im Kontext steht, sage ehrlich, dass die Information nicht vorliegt.

KONTEXT:
{context}

FRAGE:
{query}

ANTWORT (auf Deutsch, kurz und präzise):
"""

    llm = ChatOllama(model=model_name, temperature=0.1)
    response = llm.invoke(prompt)

    return {
        "answer": response.content,
        "sources": [
            {
                "source": d.metadata.get("source"),
                "page": d.metadata.get("page"),
                "filepath": d.metadata.get("filepath"),
            }
            for d in docs
        ],
    }


In [12]:
from pathlib import Path

ROOT = Path().resolve()
ARTIFACTS_DIR = ROOT / "artifacts"
print("ARTIFACTS_DIR:", ARTIFACTS_DIR)
print("Existiert:", ARTIFACTS_DIR.exists())

if ARTIFACTS_DIR.exists():
    for sub in ARTIFACTS_DIR.iterdir():
        print("Unterordner:", sub)
        if sub.is_dir():
            for f in sub.iterdir():
                print("   -", f.name)


ARTIFACTS_DIR: C:\Users\maumo\OneDrive\Dokumente\rag_project\artifacts
Existiert: True
Unterordner: C:\Users\maumo\OneDrive\Dokumente\rag_project\artifacts\chroma_din
   - bd1abfdc-bac6-47f7-af1f-dead87ec0493
   - chroma.sqlite3


In [ ]:
query = "Was ist die Anordnung der Plätze?"

retriever = vs.as_retriever(search_kwargs={"k": 4})
docs = retriever.invoke(query)

print("Anzahl gefundener Dokumente:", len(docs))
for i, d in enumerate(docs):
    print(f"\n--- Doc {i} ---")
    print("Source:", d.metadata.get("source"))
    print("Page:", d.metadata.get("page"))
    print(d.page_content[:400], "...")


Anzahl gefundener Dokumente: 3

--- Doc 0 ---
Source: din_5566_2.pdf
Page: 5
DIN 5566-2:2020-05  5  4 Plätze für das Personal  4.1 Allgemeines  Die Führerräume müssen  so aus geführt werden , dass Einmannbedienung sicherges tellt ist und dass der  Eisenbahnfahrzeugführer das Fahrzeug sowohl im Sitzen als auch im Stehen führen kann. Bei besonderen  Einsatzbedingungen kann davon abgewichen werden.  Weiterhin muss im Führerraum eine Begleitersitzgelegenheit vor gesehen werden ...

--- Doc 1 ---
Source: din_5566_2.pdf
Page: 5
Sitzplatz darf in seinen  Merkmalen von den Anforderungen nach DIN 5566-1:2020-05, 6.1 , abweichen. Das gilt nicht, wenn der  Begleiterplatz als ständiger Arbeitsplatz vorgesehen ist.  4.3 Zusatzanforderungen für Mittelführerräume  Die Anordnung des Fahrzeugführerplatzes (Standort des Führertisches bzw. -sitzes) muss hier innerhalb des  Führerraums so erfolgen, dass optimale Voraussetzungen für di ...

--- Doc 2 ---
Source: din_5566_2.pdf
Page: 2
DIN 5566-2:2020-05  2

In [17]:
query = "Was ist die Anordnung der Plätze?"
rag_result = answer_with_rag(query, vs)

print("Frage:", query)
print("\nAntwort:\n", rag_result["answer"])
print("\nQuellen:")
for s in rag_result["sources"]:
    print(f"- {s['source']} (Seite {s['page']}) → {s['filepath']}")


Frage: Was ist die Anordnung der Plätze?

Antwort:
 Die Anordnung der Plätze muss so erfolgen, dass Einmannbedienung sichergestellt ist und der Eisenbahnfahrzeugführer das Fahrzeug sowohl im Sitzen als auch im Stehen führen kann. Der Platz des Eisenbahnfahrzeugführers darf mittig oder außermittig angeordnet werden.

Quellen:
- din_5566_2.pdf (Seite 5) → C:\Users\maumo\OneDrive\Dokumente\rag_project\data\raw\din_5566_2.pdf
- din_5566_2.pdf (Seite 5) → C:\Users\maumo\OneDrive\Dokumente\rag_project\data\raw\din_5566_2.pdf
- din_5566_2.pdf (Seite 2) → C:\Users\maumo\OneDrive\Dokumente\rag_project\data\raw\din_5566_2.pdf
- din_5566_2.pdf (Seite 2) → C:\Users\maumo\OneDrive\Dokumente\rag_project\data\raw\din_5566_2.pdf


In [ ]:
from src.indexing import load_index
from src.rag import answer_with_rag, debug_retrieval

vs = load_index()

query = "Was ist der Anwendungsbereich dieser Norm DIN 5566-2?"

# Debug (optional):
debug_retrieval(query, vs, k=3)

# RAG-Antwort:
res = answer_with_rag(query, vs)
print("Antwort:\n", res["answer"])
print("\nQuellen:")
for s in res["sources"]:
    print(f"- {s['source']} (Seite {s['page']}) → {s['filepath']}")
